## Particle in a Box for Conjugated Dyes

This section builds an interactive 1D particle-in-a-box model for a single electron.

Model assumptions and units:
- Length input is in Angstrom, with allowed range 1 to 30 Angstrom.
- Energy values are computed and reported in Joules.
- The plot y-axis upper limit is user-controlled via a maximum energy input.
- Electron filling assumes spin pairing (2 electrons per level), so pi-electron count should be even.

Suggested additional library (optional):
- `IPython.display` for richer formatted output.

In [58]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.optimize import brentq
from IPython.display import display, Markdown

In [59]:
# Physical constants (SI)
h = 6.62607015e-34      # Planck constant, J*s
m_e = 9.1093837015e-31  # Electron mass, kg
c = 2.99792458e8        # Speed of light, m/s

ANGSTROM_TO_M = 1.0e-10


def pib_energies_joules(length_angstrom, n_levels):
    """Return quantum numbers and 1D PIB energies in Joules."""
    L_m = length_angstrom * ANGSTROM_TO_M
    n = np.arange(1, n_levels + 1)
    E_n = (n**2 * h**2) / (8.0 * m_e * L_m**2)
    return n, E_n


def max_quantum_number_for_energy(length_angstrom, energy_max_j):
    """Largest integer n with E_n <= energy_max_j for a given box length."""
    if energy_max_j <= 0:
        return 0

    L_m = length_angstrom * ANGSTROM_TO_M
    n_max = np.sqrt((8.0 * m_e * (L_m**2) * energy_max_j) / (h**2))
    return int(np.floor(n_max))


def homo_lumo_info(length_angstrom, n_levels, n_pi_electrons):
    """Compute HOMO/LUMO indices, energies, gap, and wavelength in nm."""
    if n_pi_electrons % 2 != 0:
        raise ValueError("pi-electron count must be even for this model")

    homo_n = n_pi_electrons // 2
    lumo_n = homo_n + 1

    if n_levels < lumo_n:
        raise ValueError(
            f"Increase displayed levels to at least {lumo_n} to include the LUMO"
        )

    n, E_n = pib_energies_joules(length_angstrom, n_levels)
    E_homo = E_n[homo_n - 1]
    E_lumo = E_n[lumo_n - 1]
    delta_E = E_lumo - E_homo

    wavelength_m = (h * c) / delta_E
    wavelength_nm = wavelength_m * 1.0e9

    return {
        "homo_n": homo_n,
        "lumo_n": lumo_n,
        "E_homo_J": E_homo,
        "E_lumo_J": E_lumo,
        "delta_E_J": delta_E,
        "wavelength_nm": wavelength_nm,
        "n": n,
        "E_n": E_n,
    }

In [60]:
# User controls for the energy-level display
length_widget = widgets.FloatSlider(
    value=12.0,
    min=1.0,
    max=30.0,
    step=0.5,
    description="L (Angstrom)",
    continuous_update=False,
)

energy_max_widget = widgets.FloatLogSlider(
    value=2.0e-18,
    base=10,
    min=-20,
    max=-16,
    step=0.05,
    description="E max (J)",
    readout_format=".2e",
    continuous_update=False,
)


def plot_energy_levels(length_angstrom, energy_max_j):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    if n_levels < 1:
        print("Increase E max: no energy levels fit in the current y-axis range.")
        return

    n, E_n = pib_energies_joules(length_angstrom, n_levels)

    fig, ax = plt.subplots(figsize=(7, 8))

    # Box is centered at x=0 with walls at -L/2 and +L/2.
    x_left, x_right = -length_angstrom / 2.0, length_angstrom / 2.0
    x_fixed_half_range = 15.5

    ax.plot([x_left, x_left], [0, energy_max_j], color="black", linewidth=2)
    ax.plot([x_right, x_right], [0, energy_max_j], color="black", linewidth=2)

    for i, energy in enumerate(E_n):
        ax.hlines(energy, x_left, x_right, colors="tab:blue", linewidth=2)
        ax.text(
            min(x_right + 0.35, x_fixed_half_range - 0.8),
            energy,
            f"n={n[i]}",
            va="center",
            fontsize=9,
        )

    ax.set_xlim(-x_fixed_half_range, x_fixed_half_range)
    ax.set_ylim(0.0, energy_max_j)
    ax.set_xlabel("Position (Angstrom)")
    ax.set_ylabel("Energy (J)")
    ax.set_title(
        f"1D Particle-in-a-Box Energy Levels (L = {length_angstrom:.1f} Angstrom, E max = {energy_max_j:.2e} J)"
    )
    plt.show()


display(Markdown("### Energy Level Display"))
display(widgets.HBox([length_widget, energy_max_widget]))
widgets.interactive_output(
    plot_energy_levels,
    {"length_angstrom": length_widget, "energy_max_j": energy_max_widget},
)

### Energy Level Display

Output()

In [61]:
# Optional table view: run this cell if you want a live table of displayed levels.
def show_energy_table(length_angstrom, energy_max_j):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    if n_levels < 1:
        display(Markdown("### Optional Table View"))
        display(Markdown("No levels fall at or below the selected maximum energy."))
        return

    n, E_n = pib_energies_joules(length_angstrom, n_levels)

    lines = ["| n | Energy (J) |", "|---:|---:|"]
    lines.extend([f"| {ni} | {energy:.6e} |" for ni, energy in zip(n, E_n)])

    display(Markdown("### Optional Table View"))
    display(Markdown("\n".join(lines)))


table_out = widgets.interactive_output(
    show_energy_table,
    {"length_angstrom": length_widget, "energy_max_j": energy_max_widget},
)
display(table_out)

Output()

In [62]:
pi_electrons_widget = widgets.IntSlider(
    value=10,
    min=2,
    max=60,
    step=2,
    description="pi e-",
    continuous_update=False,
)


def show_homo_lumo(length_angstrom, energy_max_j, n_pi_electrons):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    if n_levels < 1:
        print("Input issue: increase E max so at least one energy level is visible.")
        return

    try:
        result = homo_lumo_info(length_angstrom, n_levels, n_pi_electrons)
    except ValueError as exc:
        print(f"Input issue: {exc}")
        return

    print(f"Length, L: {length_angstrom:.2f} Angstrom")
    print(f"Maximum displayed energy: {energy_max_j:.2e} J")
    print(f"Displayed levels from E max: {n_levels}")
    print(f"pi-electrons: {n_pi_electrons}")
    print()
    print(f"HOMO level: n = {result['homo_n']}")
    print(f"LUMO level: n = {result['lumo_n']}")
    print(f"E_HOMO: {result['E_homo_J']:.6e} J")
    print(f"E_LUMO: {result['E_lumo_J']:.6e} J")
    print(f"Delta E (HOMO->LUMO): {result['delta_E_J']:.6e} J")
    print(f"Predicted transition wavelength: {result['wavelength_nm']:.2f} nm")


display(Markdown("### HOMO-LUMO Information"))
display(widgets.HBox([length_widget, energy_max_widget, pi_electrons_widget]))
widgets.interactive_output(
    show_homo_lumo,
    {
        "length_angstrom": length_widget,
        "energy_max_j": energy_max_widget,
        "n_pi_electrons": pi_electrons_widget,
    },
)

### HOMO-LUMO Information

Output()

In [63]:
# Quick verification examples
show_homo_lumo(length_angstrom=12.0, energy_max_j=2.0e-18, n_pi_electrons=10)

energy_max_verify = 5.0e-18
n_small = max_quantum_number_for_energy(8.0, energy_max_verify)
n_large = max_quantum_number_for_energy(16.0, energy_max_verify)
r_small = homo_lumo_info(length_angstrom=8.0, n_levels=n_small, n_pi_electrons=10)
r_large = homo_lumo_info(length_angstrom=16.0, n_levels=n_large, n_pi_electrons=10)

print()
print("Trend check (fixed electrons):")
print(
    f"L=8.0 Angstrom -> Delta E={r_small['delta_E_J']:.3e} J, lambda={r_small['wavelength_nm']:.1f} nm"
)
print(
    f"L=16.0 Angstrom -> Delta E={r_large['delta_E_J']:.3e} J, lambda={r_large['wavelength_nm']:.1f} nm"
)
if r_large['delta_E_J'] < r_small['delta_E_J'] and r_large['wavelength_nm'] > r_small['wavelength_nm']:
    print("Sanity check passed: larger box length gives smaller gap and longer wavelength.")
else:
    print("Sanity check failed: re-check formulas/units.")

Length, L: 12.00 Angstrom
Maximum displayed energy: 2.00e-18 J
Displayed levels from E max: 6
pi-electrons: 10

HOMO level: n = 5
LUMO level: n = 6
E_HOMO: 1.045949e-18 J
E_LUMO: 1.506167e-18 J
Delta E (HOMO->LUMO): 4.602176e-19 J
Predicted transition wavelength: 431.63 nm

Trend check (fixed electrons):
L=8.0 Angstrom -> Delta E=1.035e-18 J, lambda=191.8 nm
L=16.0 Angstrom -> Delta E=2.589e-19 J, lambda=767.3 nm
Sanity check passed: larger box length gives smaller gap and longer wavelength.


## Particle in a Box with Finite Walls

This section models a finite square well with the same centered box geometry.

Key differences from the infinite-wall model:
- The selected energy is now the **well depth** in Joules.
- Only **bound** energy levels are shown, so every displayed level satisfies $E < V_0$.
- If the well is too shallow to bind enough states, the HOMO-LUMO cell reports that clearly.

In [64]:
HBAR = h / (2.0 * np.pi)


def finite_well_bound_states(length_angstrom, well_depth_j):
    """Return all bound-state energies for a symmetric finite square well."""
    if well_depth_j <= 0:
        return {
            "n": np.array([], dtype=int),
            "parity": [],
            "E_n": np.array([]),
            "xi": np.array([]),
            "half_width_m": 0.0,
        }

    half_width_m = 0.5 * length_angstrom * ANGSTROM_TO_M
    eta = half_width_m * np.sqrt(2.0 * m_e * well_depth_j) / HBAR
    epsilon = 1.0e-9
    roots = []
    parities = []

    def even_equation(xi):
        return xi * np.tan(xi) - np.sqrt(np.maximum(eta**2 - xi**2, 0.0))

    def odd_equation(xi):
        return -xi / np.tan(xi) - np.sqrt(np.maximum(eta**2 - xi**2, 0.0))

    def add_root(func, left, right, parity):
        if left >= right:
            return
        try:
            f_left = func(left)
            f_right = func(right)
        except FloatingPointError:
            return
        if not (np.isfinite(f_left) and np.isfinite(f_right)):
            return
        if f_left == 0:
            roots.append(left)
            parities.append(parity)
            return
        if f_left * f_right > 0:
            return
        roots.append(brentq(func, left, right))
        parities.append(parity)

    max_interval = int(np.ceil(eta / np.pi)) + 2
    for interval_index in range(max_interval):
        even_left = interval_index * np.pi + epsilon
        even_right = min(interval_index * np.pi + np.pi / 2.0 - epsilon, eta - epsilon)
        add_root(even_equation, even_left, even_right, "even")

        odd_left = interval_index * np.pi + np.pi / 2.0 + epsilon
        odd_right = min((interval_index + 1) * np.pi - epsilon, eta - epsilon)
        add_root(odd_equation, odd_left, odd_right, "odd")

    if not roots:
        return {
            "n": np.array([], dtype=int),
            "parity": [],
            "E_n": np.array([]),
            "xi": np.array([]),
            "half_width_m": half_width_m,
        }

    root_array = np.array(roots)
    energies = (HBAR**2 * root_array**2) / (2.0 * m_e * half_width_m**2)
    order = np.argsort(energies)
    energies = energies[order]
    xi_sorted = root_array[order]
    parity_sorted = [parities[index] for index in order]
    state_numbers = np.arange(1, len(energies) + 1)

    return {
        "n": state_numbers,
        "parity": parity_sorted,
        "E_n": energies,
        "xi": xi_sorted,
        "half_width_m": half_width_m,
    }


def finite_well_wavefunction(length_angstrom, well_depth_j, n_state, x_angstrom):
    """Return normalized finite-well bound-state wavefunction values psi(x)."""
    states = finite_well_bound_states(length_angstrom, well_depth_j)
    if len(states["E_n"]) == 0:
        raise ValueError("No bound states exist for the selected well depth and length")
    if n_state < 1 or n_state > len(states["E_n"]):
        raise ValueError(f"State n={n_state} is outside the available bound-state range")

    idx = n_state - 1
    E_n = states["E_n"][idx]
    parity = states["parity"][idx]
    half_width_m = states["half_width_m"]

    x_m = np.asarray(x_angstrom) * ANGSTROM_TO_M
    abs_x = np.abs(x_m)

    k = np.sqrt(2.0 * m_e * E_n) / HBAR
    kappa = np.sqrt(2.0 * m_e * (well_depth_j - E_n)) / HBAR

    inside = abs_x <= half_width_m
    psi = np.zeros_like(x_m, dtype=float)

    if parity == "even":
        psi[inside] = np.cos(k * x_m[inside])
        psi[~inside] = np.cos(k * half_width_m) * np.exp(-kappa * (abs_x[~inside] - half_width_m))
    else:
        psi[inside] = np.sin(k * x_m[inside])
        psi[~inside] = (
            np.sign(x_m[~inside])
            * np.sin(k * half_width_m)
            * np.exp(-kappa * (abs_x[~inside] - half_width_m))
        )

    norm = np.trapezoid(psi**2, x_m)
    if norm > 0:
        psi = psi / np.sqrt(norm)

    return psi


def finite_well_homo_lumo_info(length_angstrom, well_depth_j, n_pi_electrons):
    """Compute HOMO/LUMO metrics for the finite well using only bound states."""
    if n_pi_electrons % 2 != 0:
        raise ValueError("pi-electron count must be even for this model")

    states = finite_well_bound_states(length_angstrom, well_depth_j)
    n_states = len(states["E_n"])
    if n_states == 0:
        raise ValueError("No bound states exist for the selected well depth and box length")

    homo_n = n_pi_electrons // 2
    lumo_n = homo_n + 1
    if n_states < lumo_n:
        raise ValueError(
            f"Only {n_states} bound states exist; need at least {lumo_n} to include the LUMO"
        )

    E_homo = states["E_n"][homo_n - 1]
    E_lumo = states["E_n"][lumo_n - 1]
    delta_E = E_lumo - E_homo
    wavelength_nm = (h * c / delta_E) * 1.0e9

    return {
        "homo_n": homo_n,
        "lumo_n": lumo_n,
        "E_homo_J": E_homo,
        "E_lumo_J": E_lumo,
        "delta_E_J": delta_E,
        "wavelength_nm": wavelength_nm,
        "n": states["n"],
        "E_n": states["E_n"],
        "parity": states["parity"],
        "n_bound": n_states,
    }

In [ ]:
# User controls for the finite-wall energy-level display
finite_length_widget = widgets.FloatSlider(
    value=12.0,
    min=1.0,
    max=30.0,
    step=0.5,
    description="L (Angstrom)",
    continuous_update=False,
)

well_depth_widget = widgets.FloatLogSlider(
    value=2.0e-18,
    base=10,
    min=-20,
    max=-16,
    step=0.05,
    description="Depth (J)",
    readout_format=".2e",
    continuous_update=False,
)

finite_selected_level_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=1,
    step=1,
    description="State n",
    continuous_update=False,
)

finite_show_state_widget = widgets.Checkbox(
    value=False,
    description="Show selected state",
)

finite_view_widget = widgets.ToggleButtons(
    options=[("Wavefunction psi", "psi"), ("Probability |psi|^2", "prob")],
    value="psi",
    description="View",
)


def plot_finite_well_levels(length_angstrom, well_depth_j, selected_n, show_state):
    states = finite_well_bound_states(length_angstrom, well_depth_j)
    n_bound = len(states["E_n"])

    if n_bound < 1:
        finite_selected_level_widget.max = 1
        finite_selected_level_widget.value = 1
        print("Increase the well depth: no bound states exist for the current settings.")
        return

    finite_selected_level_widget.max = n_bound
    if finite_selected_level_widget.value > n_bound:
        finite_selected_level_widget.value = n_bound

    fig, ax = plt.subplots(figsize=(7, 8))
    x_left, x_right = -length_angstrom / 2.0, length_angstrom / 2.0
    x_fixed_half_range = 15.5
    y_top = well_depth_j * 1.05

    ax.plot([-x_fixed_half_range, x_left], [well_depth_j, well_depth_j], color="black", linewidth=2)
    ax.plot([x_left, x_left], [0.0, well_depth_j], color="black", linewidth=2)
    ax.plot([x_right, x_right], [0.0, well_depth_j], color="black", linewidth=2)
    ax.plot([x_right, x_fixed_half_range], [well_depth_j, well_depth_j], color="black", linewidth=2)

    for state_index, energy in enumerate(states["E_n"]):
        state_n = int(states["n"][state_index])
        color = "tab:red" if show_state and state_n == selected_n else "tab:green"
        line, = ax.plot([x_left, x_right], [energy, energy], color=color, linewidth=2)
        line.set_picker(5)
        line._state_n = state_n
        ax.text(
            min(x_right + 0.35, x_fixed_half_range - 1.2),
            energy,
            f"n={state_n}",
            va="center",
            fontsize=9,
        )

    def on_pick(event):
        artist = event.artist
        picked_n = getattr(artist, "_state_n", None)
        if picked_n is not None:
            finite_selected_level_widget.value = int(picked_n)
            finite_show_state_widget.value = True

    fig.canvas.mpl_connect("pick_event", on_pick)

    ax.axhline(well_depth_j, color="gray", linestyle="--", linewidth=1)
    ax.set_xlim(-x_fixed_half_range, x_fixed_half_range)
    ax.set_ylim(0.0, y_top)
    ax.set_xlabel("Position (Angstrom)")
    ax.set_ylabel("Energy (J)")
    ax.set_title(
        f"Finite-Wall Particle-in-a-Box Bound States (L = {length_angstrom:.1f} Angstrom, depth = {well_depth_j:.2e} J)"
    )
    plt.show()


x_wave_angstrom = np.linspace(-15.5, 15.5, 1500)


def plot_selected_state(length_angstrom, well_depth_j, selected_n, show_state, view_mode):
    states = finite_well_bound_states(length_angstrom, well_depth_j)
    n_bound = len(states["E_n"])

    if n_bound < 1:
        print("No bound states available for wavefunction display.")
        return

    if selected_n > n_bound:
        print(f"Selected state n={selected_n} is not bound at current depth.")
        return

    if not show_state:
        print("Click an energy level (or check 'Show selected state') to display a state profile.")
        return

    psi = finite_well_wavefunction(length_angstrom, well_depth_j, selected_n, x_wave_angstrom)
    y = psi if view_mode == "psi" else psi**2

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(x_wave_angstrom, y, color="tab:purple", linewidth=2)
    x_left, x_right = -length_angstrom / 2.0, length_angstrom / 2.0
    ax.axvline(x_left, color="black", linestyle="--", linewidth=1)
    ax.axvline(x_right, color="black", linestyle="--", linewidth=1)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_xlim(-15.5, 15.5)
    ax.set_xlabel("Position (Angstrom)")
    if view_mode == "psi":
        ax.set_ylabel("psi(x)")
        ax.set_title(f"Finite-Wall Wavefunction for n={selected_n}")
    else:
        ax.set_ylabel("|psi(x)|^2")
        ax.set_title(f"Finite-Wall Probability Distribution for n={selected_n}")
    plt.show()


display(Markdown("### Finite-Wall Energy Level Display"))
display(widgets.HBox([finite_length_widget, well_depth_widget]))
display(Markdown("Click an energy line to choose a state for the profile view below."))
display(widgets.HBox([finite_selected_level_widget, finite_show_state_widget, finite_view_widget]))

finite_levels_out = widgets.interactive_output(
    plot_finite_well_levels,
    {
        "length_angstrom": finite_length_widget,
        "well_depth_j": well_depth_widget,
        "selected_n": finite_selected_level_widget,
        "show_state": finite_show_state_widget,
    },
)

finite_state_out = widgets.interactive_output(
    plot_selected_state,
    {
        "length_angstrom": finite_length_widget,
        "well_depth_j": well_depth_widget,
        "selected_n": finite_selected_level_widget,
        "show_state": finite_show_state_widget,
        "view_mode": finite_view_widget,
    },
)

display(finite_levels_out)
display(finite_state_out)

### Finite-Wall Energy Level Display

Click an energy line to choose a state for the profile view below.

Output()

Output()

In [66]:
# Optional table view for the finite well.
def show_finite_well_table(length_angstrom, well_depth_j):
    states = finite_well_bound_states(length_angstrom, well_depth_j)
    display(Markdown("### Optional Finite-Wall Table View"))
    if len(states["E_n"]) == 0:
        display(Markdown("No bound states fall below the selected well depth."))
        return

    lines = ["| n | Parity | Energy (J) |", "|---:|:---:|---:|"]
    for state_number, parity, energy in zip(states["n"], states["parity"], states["E_n"]):
        lines.append(f"| {state_number} | {parity} | {energy:.6e} |")
    display(Markdown("\n".join(lines)))


finite_table_out = widgets.interactive_output(
    show_finite_well_table,
    {"length_angstrom": finite_length_widget, "well_depth_j": well_depth_widget},
)
display(finite_table_out)

Output()

In [67]:
finite_pi_electrons_widget = widgets.IntSlider(
    value=10,
    min=2,
    max=60,
    step=2,
    description="pi e-",
    continuous_update=False,
)


def show_finite_well_homo_lumo(length_angstrom, well_depth_j, n_pi_electrons):
    try:
        result = finite_well_homo_lumo_info(length_angstrom, well_depth_j, n_pi_electrons)
    except ValueError as exc:
        print(f"Input issue: {exc}")
        return

    print(f"Length, L: {length_angstrom:.2f} Angstrom")
    print(f"Well depth: {well_depth_j:.2e} J")
    print(f"Bound states available: {result["n_bound"]}")
    print(f"pi-electrons: {n_pi_electrons}")
    print()
    print(f"HOMO level: n = {result["homo_n"]}")
    print(f"LUMO level: n = {result["lumo_n"]}")
    print(f"E_HOMO: {result["E_homo_J"]:.6e} J")
    print(f"E_LUMO: {result["E_lumo_J"]:.6e} J")
    print(f"Delta E (HOMO->LUMO): {result["delta_E_J"]:.6e} J")
    print(f"Predicted transition wavelength: {result["wavelength_nm"]:.2f} nm")


display(Markdown("### Finite-Wall HOMO-LUMO Information"))
display(widgets.HBox([finite_length_widget, well_depth_widget, finite_pi_electrons_widget]))
widgets.interactive_output(
    show_finite_well_homo_lumo,
    {
        "length_angstrom": finite_length_widget,
        "well_depth_j": well_depth_widget,
        "n_pi_electrons": finite_pi_electrons_widget,
    },
)

### Finite-Wall HOMO-LUMO Information

Output()

In [68]:
# Quick finite-well verification example
finite_states_shallow = finite_well_bound_states(12.0, 1.0e-18)
finite_states_deep = finite_well_bound_states(12.0, 3.0e-18)
print(f"Bound states at depth 1.0e-18 J: {len(finite_states_shallow["E_n"])}")
print(f"Bound states at depth 3.0e-18 J: {len(finite_states_deep["E_n"])}")
if len(finite_states_deep["E_n"]) >= len(finite_states_shallow["E_n"]):
    print("Sanity check passed: deeper wells support at least as many bound states.")
else:
    print("Sanity check failed: re-check the finite-well solver.")

Bound states at depth 1.0e-18 J: 5
Bound states at depth 3.0e-18 J: 9
Sanity check passed: deeper wells support at least as many bound states.
